# MSDS 565 — Unit 1, Notebook 4: Feature Selection

**Team DataStars**

Use Random Forest feature importance and VIF filtering to reduce the feature set, then retrain and compare model performance before and after feature selection.

## 1. Setup

### 1.1 Setup and Imports

Import the libraries needed for data handling, feature selection, model training, evaluation, model saving, and neural networks.

In [1]:
# Core libraries
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Scikit-learn models and utilities
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

# Model saving
import joblib

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

### 1.2 Load Processed Training and Test Data

Load the processed training and test datasets created in Notebook 2. Using the same 80/20 split from the previous notebooks ensures that the feature-selection models can be compared fairly with the baseline models from Notebook 3.

In [2]:
# Path to the project data folder
data_dir = Path("../../data")

# Load the same processed datasets used in Notebook 3
train_df = pd.read_csv(data_dir / "processed_train.csv")
test_df = pd.read_csv(data_dir / "processed_test.csv")

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (237268, 312)
Test data shape: (59317, 312)


### 1.3 Define the Target

The regression target is `price`. Separate the target variable from the training and test datasets so that it is not included among the predictor features.

In [3]:
# Define the target variable
target = "price"

y_train = train_df[target].copy()
y_test = test_df[target].copy()

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target summary:")
print(y_train.describe())

y_train shape: (237268,)
y_test shape: (59317,)

Training target summary:
count    2.372680e+05
mean     2.914459e+04
std      2.070670e+04
min      7.940000e+02
25%      1.722100e+04
50%      2.509000e+04
75%      3.720200e+04
max      1.750000e+06
Name: price, dtype: float64


### 1.4 Identify Metadata Columns

Identify the object-dtype columns retained as metadata during preprocessing. These columns are preserved for the fairness analysis in Notebook 5 but are excluded from the feature matrices used for modeling and feature selection.

In [4]:
# Identify object-dtype metadata columns
metadata_cols = train_df.select_dtypes(include="object").columns.tolist()

print("Number of metadata columns:", len(metadata_cols))
print("\nMetadata columns:")
for col in metadata_cols:
    print(f"  - {col}")

Number of metadata columns: 18

Metadata columns:
  - city
  - bed
  - body_type
  - cabin
  - engine_type
  - exterior_color
  - franchise_make
  - fuel_type
  - interior_color
  - listing_color
  - major_options
  - make_name
  - model_name
  - sp_name
  - transmission
  - transmission_display
  - trim_name
  - wheel_system


### 1.5 Construct Modeling Feature Matrices

Construct the training and test feature matrices by excluding the `price` target and all object-dtype metadata columns. This preserves the metadata in the original datasets for later fairness analysis while ensuring that only model-ready numeric features are used for feature selection and modeling.

In [5]:
# Columns to exclude from modeling
exclude_cols = [target] + metadata_cols

# Construct feature matrices
X_train = train_df.drop(columns=exclude_cols).copy()
X_test = test_df.drop(columns=exclude_cols).copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nNumber of modeling features:", X_train.shape[1])

X_train shape: (237268, 293)
X_test shape: (59317, 293)

y_train shape: (237268,)
y_test shape: (59317,)

Number of modeling features: 293


### 1.6 Load Notebook 3 Baseline Model Metrics

Load the model performance metrics exported from Notebook 3. These results serve as the baseline for comparing model accuracy and training time before and after feature selection.

In [6]:
# Load Notebook 3 baseline model performance
baseline_metrics = pd.read_csv(
    data_dir / "model_metrics.csv",
    index_col=0
)

print("Notebook 3 Baseline Model Performance:")
display(baseline_metrics)

Notebook 3 Baseline Model Performance:


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,2.370951
Random Forest,1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
Ridge Regression,5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
Keras Architecture 2,2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758


### 1.7 Modeling Data Quality Check

Verify that the training and test feature matrices contain the same features and consist only of numeric, finite, non-missing values. This check ensures that the processed data are ready for Random Forest importance ranking and subsequent VIF analysis.

In [7]:
# Check that train and test contain identical feature columns
columns_match = X_train.columns.equals(X_test.columns)

# Check for non-numeric columns
non_numeric_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Check for missing values
train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

# Check for infinite values
train_inf = np.isinf(X_train.to_numpy()).sum()
test_inf = np.isinf(X_test.to_numpy()).sum()

print("Modeling Data Quality Check")
print("-" * 35)

print("Train/test columns match:", columns_match)
print("Non-numeric feature columns:", len(non_numeric_cols))
print("Missing values in X_train:", train_missing)
print("Missing values in X_test:", test_missing)
print("Infinite values in X_train:", train_inf)
print("Infinite values in X_test:", test_inf)

if non_numeric_cols:
    print("\nNon-numeric columns found:")
    print(non_numeric_cols)

Modeling Data Quality Check
-----------------------------------
Train/test columns match: True
Non-numeric feature columns: 0
Missing values in X_train: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0


### Section 1 Summary

The processed training and test datasets from Notebook 2 were successfully loaded using the same 80/20 split used in Notebook 3. The training set contains 237,268 observations and the test set contains 59,317 observations, with 312 columns in each dataset.

The `price` variable was separated as the regression target. Eighteen object-dtype columns were identified as metadata and preserved for the fairness analysis in Notebook 5 but excluded from modeling. After removing the target and metadata columns, the modeling datasets contain 293 numeric predictor features.

Notebook 3's model performance results were also loaded to provide the baseline for the before-and-after feature-selection comparison. Finally, the modeling feature matrices were checked to confirm that the training and test datasets have matching feature structures and are ready for feature selection.

## 2. Random Forest Importance-Based Feature Selection

Random Forest feature importance is used as the first stage of feature selection. The model evaluates how useful each predictor is for reducing prediction error across its decision trees. Features are then ranked according to their importance, and the 100 most important features are retained for the next stage of feature selection.

### 2.1 Fit Random Forest for Feature Importance

Fit a Random Forest regressor using the full set of 293 modeling features in the training data. This model is used specifically to calculate feature importance and reduce the feature set before VIF filtering.

In [8]:
# Random Forest used for feature-importance ranking
rf_importance = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Fit using the full training feature set
start_time = time.time()

rf_importance.fit(X_train, y_train)

importance_fit_time = time.time() - start_time

print("Random Forest feature-importance model fitted successfully.")
print(f"Features evaluated: {X_train.shape[1]}")
print(f"Training observations: {X_train.shape[0]:,}")
print(f"Fit time: {importance_fit_time:.2f} seconds")

Random Forest feature-importance model fitted successfully.
Features evaluated: 293
Training observations: 237,268
Fit time: 100.33 seconds


### 2.2 Rank Features by Importance

Extract the feature importance scores from the fitted Random Forest and rank all 293 predictor features from highest to lowest importance. Higher importance scores indicate that a feature contributed more to the Random Forest's predictions.

In [9]:
# Create a table of feature importance scores
feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_importance.feature_importances_
})

# Rank features from most to least important
feature_importance_df = (
    feature_importance_df
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# Add ranking
feature_importance_df.index = feature_importance_df.index + 1
feature_importance_df.index.name = "Rank"

print("Top 20 Features by Random Forest Importance:")
display(feature_importance_df.head(20))

Top 20 Features by Random Forest Importance:


,Feature,Importance
Rank,,
1,horsepower,0.466860
2,mileage,0.204802
3,height,0.047775
4,engine_displacement,0.029004
5,vehicle_age,0.020446
6,opt_sunroof_moonroof,0.019716
7,fuel_tank_volume,0.016310
8,length,0.016034
9,width,0.013668


### 2.3 Select the Top 100 Features

Retain the 100 highest-ranked features based on Random Forest feature importance. This reduces the original 293-feature modeling set while preserving the predictors that the Random Forest identified as most useful for predicting vehicle price.

In [10]:
# Select the 100 most important features
top_100_features = feature_importance_df.head(100)["Feature"].tolist()

# Create importance-reduced training and test feature matrices
X_train_top100 = X_train[top_100_features].copy()
X_test_top100 = X_test[top_100_features].copy()

print("Original number of features:", X_train.shape[1])
print("Selected number of features:", len(top_100_features))

print("\nX_train_top100 shape:", X_train_top100.shape)
print("X_test_top100 shape:", X_test_top100.shape)

Original number of features: 293
Selected number of features: 100

X_train_top100 shape: (237268, 100)
X_test_top100 shape: (59317, 100)


### 2.4 Examine Selected Features

Examine the highest-ranked features retained by the Random Forest and calculate how much of the model's total feature importance is represented by the selected top 100 features. This provides insight into how much predictive information was retained after reducing the feature set.

In [11]:
# Examine the top 25 selected features
print("Top 25 Selected Features:")
display(feature_importance_df.head(25))

# Calculate cumulative importance
top_10_importance = feature_importance_df.head(10)["Importance"].sum()
top_25_importance = feature_importance_df.head(25)["Importance"].sum()
top_100_importance = feature_importance_df.head(100)["Importance"].sum()

print("Cumulative Feature Importance")
print("-" * 35)
print(f"Top 10 features:  {top_10_importance:.2%}")
print(f"Top 25 features:  {top_25_importance:.2%}")
print(f"Top 100 features: {top_100_importance:.2%}")

Top 25 Selected Features:


,Feature,Importance
Rank,,
1,horsepower,0.466860
2,mileage,0.204802
3,height,0.047775
4,engine_displacement,0.029004
5,vehicle_age,0.020446
6,opt_sunroof_moonroof,0.019716
7,fuel_tank_volume,0.016310
8,length,0.016034
9,width,0.013668


Cumulative Feature Importance
-----------------------------------
Top 10 features:  84.82%
Top 25 features:  93.01%
Top 100 features: 99.12%


### Section 2 Summary

Random Forest feature importance was used as the first stage of feature selection. The model evaluated all 293 predictor features using the training data and ranked them according to their contribution to the Random Forest's predictions.

The importance scores were highly concentrated among a relatively small number of predictors. `horsepower` was the highest-ranked feature with an importance of 0.4669, followed by `mileage` at 0.2048. Together, these two features accounted for approximately 67.17% of the total importance assigned by the Random Forest.

The top 10 features accounted for 84.82% of total feature importance, while the top 25 accounted for 93.01%. Following the project requirements, the 100 highest-ranked features were retained. These 100 features accounted for 99.12% of the Random Forest's total feature importance, reducing the modeling feature set from 293 to 100 predictors.

The selected 100 features will now undergo VIF filtering to identify and reduce multicollinearity before the models are retrained.

## 3. VIF Filtering

Variance Inflation Factor (VIF) is used as the second stage of feature selection to identify multicollinearity among the 100 features retained through Random Forest importance. Features with excessive VIF values will be removed iteratively until the remaining feature set satisfies the VIF threshold of 10.

### 3.1 Prepare Features for VIF Analysis

Before calculating VIF, inspect the 100 importance-selected features for constant predictors and identify binary predictors. Constant features can create problems during VIF calculation, while identifying binary predictors provides additional information about the structure of the selected feature set before beginning iterative VIF filtering.

In [12]:
# Count the number of unique values in each selected feature
unique_counts = X_train_top100.nunique()

# Identify constant features
constant_features = unique_counts[unique_counts <= 1]

# Identify binary features
binary_features = unique_counts[
    (unique_counts > 1) & (unique_counts <= 2)
]

print("VIF Preparation Check")
print("-" * 35)

print("Features entering VIF analysis:", X_train_top100.shape[1])
print("Constant features:", len(constant_features))
print("Features with only 2 unique values:", len(binary_features))

if len(constant_features) > 0:
    print("\nConstant features:")
    display(constant_features.to_frame("Unique Values"))

if len(binary_features) > 0:
    print("\nBinary features:")
    display(binary_features.to_frame("Unique Values"))

VIF Preparation Check
-----------------------------------
Features entering VIF analysis: 100
Constant features: 0
Features with only 2 unique values: 76

Binary features:


,Unique Values
opt_sunroof_moonroof,2
opt_navigation_system,2
wheel_system_AWD,2
franchise_make_BMW,2
engine_type_H6,2
...,...
engine_type_I6,2
exterior_color_Unknown,2
fuel_type_Flex Fuel Vehicle,2
city_Dallas,2


### 3.2 Calculate Initial VIF Values

Calculate the initial Variance Inflation Factor (VIF) values for the 100 features retained through Random Forest importance. VIF measures how strongly each predictor can be explained by the other predictors in the feature set.

To make the calculation computationally efficient for the large training dataset, VIF values are calculated from the correlation matrix of the full training data. Features with VIF values greater than 10 are considered candidates for removal due to high multicollinearity.

In [13]:
# Calculate the correlation matrix for the 100 selected features
corr_matrix = X_train_top100.corr()

# Use the pseudo-inverse for numerical stability
inv_corr_matrix = np.linalg.pinv(corr_matrix.to_numpy())

# VIF values are the diagonal of the inverse correlation matrix
initial_vif = pd.DataFrame({
    "Feature": corr_matrix.columns,
    "VIF": np.diag(inv_corr_matrix)
})

# Sort from highest to lowest VIF
initial_vif = (
    initial_vif
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

initial_vif.index = initial_vif.index + 1
initial_vif.index.name = "Rank"

print("Initial VIF Results")
print("-" * 35)
print("Features evaluated:", len(initial_vif))
print("Features with VIF > 10:", (initial_vif["VIF"] > 10).sum())

print("\nTop 20 Highest VIF Values:")
display(initial_vif.head(20))

Initial VIF Results
-----------------------------------
Features evaluated: 100
Features with VIF > 10: 12

Top 20 Highest VIF Values:


,Feature,VIF
Rank,,
1,engine_type_Unknown,103.735132
2,fuel_type_Unknown,86.111087
3,bed_length,40.888900
4,has_bed,39.169788
5,length,29.560396
6,wheelbase,28.555959
7,is_new,23.290286
8,fuel_type_Electric,22.981840
9,highway_fuel_economy,21.413683


### 3.3 Iteratively Remove Features with VIF > 10

Apply iterative VIF filtering to the 100 importance-selected features. At each iteration, the feature with the highest VIF greater than 10 is removed, and VIF values are recalculated for the remaining features. This process continues until all remaining predictors have VIF values at or below the threshold of 10.

In [14]:
# Start with all 100 Random Forest-selected features
remaining_features = top_100_features.copy()

# Store information about each removed feature
vif_removal_log = []

iteration = 1

while True:
    # Correlation matrix for the currently retained features
    corr_matrix = X_train_top100[remaining_features].corr()

    # Pseudo-inverse for numerical stability
    inv_corr_matrix = np.linalg.pinv(corr_matrix.to_numpy())

    # Calculate VIF values
    vif_values = pd.Series(
        np.diag(inv_corr_matrix),
        index=corr_matrix.columns
    )

    # Identify the feature with the highest VIF
    max_vif_feature = vif_values.idxmax()
    max_vif = vif_values.max()

    # Stop when all remaining features have VIF <= 10
    if max_vif <= 10:
        break

    # Record the feature before removing it
    vif_removal_log.append({
        "Iteration": iteration,
        "Feature Removed": max_vif_feature,
        "VIF at Removal": max_vif
    })

    # Remove the highest-VIF feature
    remaining_features.remove(max_vif_feature)

    iteration += 1

# Convert removal history to a DataFrame
vif_removal_df = pd.DataFrame(vif_removal_log)

print("Iterative VIF Filtering Complete")
print("-" * 40)
print("Starting features:", len(top_100_features))
print("Features removed:", len(vif_removal_df))
print("Features remaining:", len(remaining_features))
print(f"Highest remaining VIF: {max_vif:.4f}")

print("\nVIF Removal History:")
display(vif_removal_df)

Iterative VIF Filtering Complete
----------------------------------------
Starting features: 100
Features removed: 6
Features remaining: 94
Highest remaining VIF: 9.0558

VIF Removal History:


,Iteration,Feature Removed,VIF at Removal
0,1,engine_type_Unknown,103.735132
1,2,bed_length,40.869532
2,3,length,29.553711
3,4,is_new,23.286528
4,5,highway_fuel_economy,21.370077
5,6,wheelbase,12.565428


### 3.4 Create Final Reduced Feature Matrices

Use the features retained after iterative VIF filtering to create the final reduced training and test feature matrices. These matrices contain the predictors that survived both Random Forest importance selection and VIF filtering and will be used to retrain the models.

In [15]:
# Create final reduced feature matrices
X_train_selected = X_train[remaining_features].copy()
X_test_selected = X_test[remaining_features].copy()

print("Final Reduced Feature Matrices")
print("-" * 40)

print("Original features:", X_train.shape[1])
print("After Random Forest selection:", X_train_top100.shape[1])
print("After VIF filtering:", X_train_selected.shape[1])

print("\nX_train_selected shape:", X_train_selected.shape)
print("X_test_selected shape:", X_test_selected.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print(
    "\nTrain/test selected columns match:",
    X_train_selected.columns.equals(X_test_selected.columns)
)

Final Reduced Feature Matrices
----------------------------------------
Original features: 293
After Random Forest selection: 100
After VIF filtering: 94

X_train_selected shape: (237268, 94)
X_test_selected shape: (59317, 94)

y_train shape: (237268,)
y_test shape: (59317,)

Train/test selected columns match: True


### 3.5 Examine Final VIF-Selected Feature Set

Verify the VIF values of the final 94-feature set after iterative filtering. The final feature set should contain no predictors with VIF values greater than 10. The features removed during VIF filtering are also summarized to document the reduction process.

In [16]:
# Calculate final VIF values for the retained features
final_corr_matrix = X_train_selected.corr()
final_inv_corr_matrix = np.linalg.pinv(final_corr_matrix.to_numpy())

final_vif = pd.DataFrame({
    "Feature": final_corr_matrix.columns,
    "VIF": np.diag(final_inv_corr_matrix)
})

# Sort from highest to lowest VIF
final_vif = (
    final_vif
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

final_vif.index = final_vif.index + 1
final_vif.index.name = "Rank"

print("Final VIF Verification")
print("-" * 40)

print("Final number of features:", len(final_vif))
print("Features with VIF > 10:", (final_vif["VIF"] > 10).sum())
print(f"Maximum VIF: {final_vif['VIF'].max():.4f}")

print("\nTop 20 Highest Remaining VIF Values:")
display(final_vif.head(20))

print("\nFeatures Removed During VIF Filtering:")
display(vif_removal_df)

Final VIF Verification
----------------------------------------
Final number of features: 94
Features with VIF > 10: 0
Maximum VIF: 9.0558

Top 20 Highest Remaining VIF Values:


,Feature,VIF
Rank,,
1,engine_displacement,9.055831
2,height,8.411418
3,horsepower,6.851243
4,mileage,6.383864
5,fuel_tank_volume,6.228441
6,n_options,6.178392
7,city_fuel_economy,5.390932
8,mileage_per_year,4.542339
9,engine_type_V8,4.426493



Features Removed During VIF Filtering:


,Iteration,Feature Removed,VIF at Removal
0,1,engine_type_Unknown,103.735132
1,2,bed_length,40.869532
2,3,length,29.553711
3,4,is_new,23.286528
4,5,highway_fuel_economy,21.370077
5,6,wheelbase,12.565428


### Section 3 Summary

VIF filtering was applied as the second stage of feature selection to address multicollinearity among the 100 features retained through Random Forest importance. The initial VIF analysis identified 12 features with VIF values greater than 10.

Rather than removing all 12 features simultaneously, VIF was recalculated after each removal. At each iteration, the feature with the highest VIF above 10 was removed. This iterative process removed six features: `engine_type_Unknown`, `bed_length`, `length`, `is_new`, `highway_fuel_economy`, and `wheelbase`.

After the six removals, 94 features remained. Final VIF verification confirmed that none of the retained features had a VIF greater than 10. The highest remaining VIF was 9.0558 for `engine_displacement`.

Across both stages of feature selection, the original 293-feature modeling set was reduced first to 100 features through Random Forest importance and then to 94 features through VIF filtering. This represents an overall reduction of approximately 67.9% in the number of predictor features. The final 94-feature set will be used to retrain and evaluate the regression models.

## 4. Retrain Scikit-Learn Models

The scikit-learn regression models from Notebook 3 are retrained using the final 94-feature dataset. The same model types, parameter grids, cross-validation strategy, and evaluation metrics are maintained so that model performance before and after feature selection can be compared consistently.

### 4.1 Define Cross-Validation Strategy and Evaluation Metrics

Use 5-fold cross-validation with shuffled observations and a fixed random state for reproducibility. Model performance is evaluated using Mean Absolute Error (MAE), Mean Absolute Percentage Error (MAPE), and R², matching the evaluation approach used in Notebook 3.

In [17]:
# Define the same 5-fold cross-validation strategy used in Notebook 3
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define scoring metrics for GridSearchCV
scoring = {
    "MAE": "neg_mean_absolute_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2"
}

print("Cross-Validation Configuration")
print("-" * 40)
print("Number of folds:", cv.get_n_splits())
print("Shuffle:", cv.shuffle)
print("Random state:", cv.random_state)

print("\nEvaluation Metrics")
print("-" * 40)
print("MAE  - Mean Absolute Error")
print("MAPE - Mean Absolute Percentage Error")
print("R²   - Coefficient of Determination")

Cross-Validation Configuration
----------------------------------------
Number of folds: 5
Shuffle: True
Random state: 42

Evaluation Metrics
----------------------------------------
MAE  - Mean Absolute Error
MAPE - Mean Absolute Percentage Error
R²   - Coefficient of Determination


### 4.2 Linear Regression Grid Search

Retrain the Linear Regression model using the final 94-feature training set. The same parameter grid used in Notebook 3 is evaluated with 5-fold GridSearchCV. MAE is used as the refit metric so that the parameter combination with the lowest cross-validation MAE is selected for the final model.

In [18]:
# Define Linear Regression model
linear_model = LinearRegression()

# Same parameter grid used in Notebook 3
linear_param_grid = {
    "fit_intercept": [True, False],
    "positive": [False, True]
}

# Set up GridSearchCV
linear_grid = GridSearchCV(
    estimator=linear_model,
    param_grid=linear_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

# Time the complete grid search and refit
start_time = time.time()

linear_grid.fit(X_train_selected, y_train)

linear_training_time = time.time() - start_time

print("Linear Regression Grid Search Complete")
print("-" * 45)
print("Best parameters:", linear_grid.best_params_)
print(f"Best CV MAE: {-linear_grid.best_score_:.4f}")
print(f"Training time: {linear_training_time:.2f} seconds")

Linear Regression Grid Search Complete
---------------------------------------------
Best parameters: {'fit_intercept': True, 'positive': False}
Best CV MAE: 5991.0166
Training time: 13.23 seconds


### 4.3 Random Forest Grid Search

Retrain the Random Forest model using the final 94-feature training set. The same hyperparameter grid used in Notebook 3 is evaluated using 5-fold GridSearchCV, with MAE used as the refit metric. This allows the reduced-feature Random Forest to be compared directly with the full-feature baseline model.

In [19]:
# Define Random Forest model
rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Same parameter grid used in Notebook 3
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.5]
}

# Set up GridSearchCV
# Limit GridSearchCV to 2 simultaneous jobs to reduce memory pressure
rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=2,
    return_train_score=False,
    verbose=1
)

# Time the complete grid search and refit
start_time = time.time()

rf_grid.fit(X_train_selected, y_train)

rf_training_time = time.time() - start_time

print("\nRandom Forest Grid Search Complete")
print("-" * 45)
print("Best parameters:", rf_grid.best_params_)
print(f"Best CV MAE: {-rf_grid.best_score_:.4f}")
print(f"Training time: {rf_training_time:.2f} seconds")

Fitting 5 folds for each of 16 candidates, totalling 80 fits

Random Forest Grid Search Complete
---------------------------------------------
Best parameters: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV MAE: 1799.8163
Training time: 2065.11 seconds


### 4.4 Ridge Regression Grid Search

Retrain the Ridge Regression model using the final 94-feature training set. The same alpha values evaluated in Notebook 3 are tested using 5-fold GridSearchCV. MAE is used as the refit metric so that the model with the lowest cross-validation MAE is selected.

In [20]:
# Define Ridge Regression model
ridge_model = Ridge()

# Same parameter grid used in Notebook 3
ridge_param_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

# Set up GridSearchCV
ridge_grid = GridSearchCV(
    estimator=ridge_model,
    param_grid=ridge_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

# Time the complete grid search and refit
start_time = time.time()

ridge_grid.fit(X_train_selected, y_train)

ridge_training_time = time.time() - start_time

print("Ridge Regression Grid Search Complete")
print("-" * 45)
print("Best parameters:", ridge_grid.best_params_)
print(f"Best CV MAE: {-ridge_grid.best_score_:.4f}")
print(f"Training time: {ridge_training_time:.2f} seconds")

Ridge Regression Grid Search Complete
---------------------------------------------
Best parameters: {'alpha': 100.0}
Best CV MAE: 5987.3316
Training time: 7.29 seconds


### 4.5 Evaluate Scikit-Learn Models on the Test Set

Extract the cross-validation results associated with the best parameter combination for each scikit-learn model and evaluate the refitted best estimators on the held-out test set. MAE, MAPE, and R² are calculated using the same evaluation metrics used in Notebook 3.

In [21]:
def evaluate_grid_model(grid, model_name, training_time):
    """
    Extract CV metrics for the best GridSearchCV model
    and calculate held-out test metrics.
    """

    # Location of the best parameter combination
    best_idx = grid.best_index_

    # Cross-validation metrics
    cv_mae = -grid.cv_results_["mean_test_MAE"][best_idx]
    cv_mape = -grid.cv_results_["mean_test_MAPE"][best_idx]
    cv_r2 = grid.cv_results_["mean_test_R2"][best_idx]

    # Best estimator was already refitted on the full training set
    best_model = grid.best_estimator_

    # Predict the held-out test set
    y_pred = best_model.predict(X_test_selected)

    # Test metrics
    test_mae = mean_absolute_error(y_test, y_pred)
    test_mape = mean_absolute_percentage_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    return {
        "Model": model_name,
        "CV MAE": cv_mae,
        "CV MAPE": cv_mape,
        "CV R2": cv_r2,
        "Test MAE": test_mae,
        "Test MAPE": test_mape,
        "Test R2": test_r2,
        "Training Time (sec)": training_time
    }

In [22]:
# Evaluate the three selected-feature scikit-learn models
sklearn_selected_results = []

sklearn_selected_results.append(
    evaluate_grid_model(
        linear_grid,
        "Linear Regression",
        linear_training_time
    )
)

sklearn_selected_results.append(
    evaluate_grid_model(
        rf_grid,
        "Random Forest",
        rf_training_time
    )
)

sklearn_selected_results.append(
    evaluate_grid_model(
        ridge_grid,
        "Ridge Regression",
        ridge_training_time
    )
)

# Create results table
sklearn_selected_metrics = (
    pd.DataFrame(sklearn_selected_results)
    .set_index("Model")
)

print("Selected-Feature Scikit-Learn Model Performance")
display(sklearn_selected_metrics)

Selected-Feature Scikit-Learn Model Performance


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5991.016587,0.288787,0.651157,5947.921657,0.291266,0.700956,13.229104
Random Forest,1799.816284,0.074237,0.898400,1701.087367,0.071589,0.947267,2065.110316
Ridge Regression,5987.331618,0.288532,0.651159,5944.907367,0.291065,0.700997,7.289425


### 4.6 Scikit-Learn Model Results

Summarize the performance of the three scikit-learn regression models after Random Forest importance selection and VIF filtering. These results represent model performance using the final 94-feature dataset and will later be compared with the corresponding full-feature results from Notebook 3.

In [23]:
# Display selected-feature scikit-learn results
sklearn_results_display = sklearn_selected_metrics.copy()

# Round values for easier interpretation
sklearn_results_display = sklearn_results_display.round({
    "CV MAE": 2,
    "CV MAPE": 4,
    "CV R2": 4,
    "Test MAE": 2,
    "Test MAPE": 4,
    "Test R2": 4,
    "Training Time (sec)": 2
})

print("Scikit-Learn Model Performance After Feature Selection")
print("-" * 60)

display(sklearn_results_display)

Scikit-Learn Model Performance After Feature Selection
------------------------------------------------------------


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5991.02,0.2888,0.6512,5947.92,0.2913,0.7010,13.23
Random Forest,1799.82,0.0742,0.8984,1701.09,0.0716,0.9473,2065.11
Ridge Regression,5987.33,0.2885,0.6512,5944.91,0.2911,0.7010,7.29


### 4.7 Final Refit Training Times

The training times reported during GridSearchCV include the time required to evaluate multiple hyperparameter combinations across five cross-validation folds. Therefore, those values do not represent the training time of a single final model.

To obtain final-fit training times for comparison, each scikit-learn model is rebuilt using its best hyperparameters and fitted once on the complete 94-feature training dataset. Only this final model fit is timed.

These final refit times will be used in the before-and-after comparison with Notebook 3.

In [24]:
# Retrieve best parameters from completed grid searches

linear_best_params = linear_grid.best_params_
rf_best_params = rf_grid.best_params_
ridge_best_params = ridge_grid.best_params_

print("Linear Regression:", linear_best_params)
print("Random Forest:", rf_best_params)
print("Ridge Regression:", ridge_best_params)

Linear Regression: {'fit_intercept': True, 'positive': False}
Random Forest: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
Ridge Regression: {'alpha': 100.0}


In [25]:
# ---------------------------------------------------------
# Linear Regression final refit
# ---------------------------------------------------------

linear_selected = LinearRegression(**linear_best_params)

start_time = time.perf_counter()

linear_selected.fit(
    X_train_selected,
    y_train
)

linear_selected_train_time = time.perf_counter() - start_time


# ---------------------------------------------------------
# Random Forest final refit
# ---------------------------------------------------------

rf_selected = RandomForestRegressor(
    **rf_best_params,
    random_state=42,
    n_jobs=-1
)

start_time = time.perf_counter()

rf_selected.fit(
    X_train_selected,
    y_train
)

rf_selected_train_time = time.perf_counter() - start_time


# ---------------------------------------------------------
# Ridge Regression final refit
# ---------------------------------------------------------

ridge_selected = Ridge(**ridge_best_params)

start_time = time.perf_counter()

ridge_selected.fit(
    X_train_selected,
    y_train
)

ridge_selected_train_time = time.perf_counter() - start_time


# Display final refit times
print("Final Refit Training Times")
print("-" * 40)
print(
    f"Linear Regression: "
    f"{linear_selected_train_time:.2f} seconds"
)
print(
    f"Random Forest: "
    f"{rf_selected_train_time:.2f} seconds"
)
print(
    f"Ridge Regression: "
    f"{ridge_selected_train_time:.2f} seconds"
)

Final Refit Training Times
----------------------------------------
Linear Regression: 0.37 seconds
Random Forest: 56.19 seconds
Ridge Regression: 0.21 seconds


**Interpretation:**  
The final refit times are substantially lower than the GridSearchCV times because the final models are trained only once using the previously selected best hyperparameters. These final refit times will be used when comparing training speed with the original Notebook 3 models.

### Section 4 Summary

The three scikit-learn regression models from Notebook 3 were retrained using the 94 features retained after Random Forest importance selection and VIF filtering. The same 5-fold cross-validation strategy and hyperparameter grids were maintained to support comparison with the original full-feature models.

Linear Regression selected `fit_intercept=True` and `positive=False`. It produced a cross-validation MAE of 5991.02 and a held-out test MAE of 5947.92, with a test R² of 0.7010.

Random Forest selected `max_depth=None`, `max_features=0.5`, `min_samples_leaf=1`, and `n_estimators=200`. It produced the lowest prediction errors among the three scikit-learn models, with a cross-validation MAE of 1799.82 and a held-out test MAE of 1701.09. Its test R² of 0.9473 indicates that the model explained approximately 94.7% of the variation in vehicle prices in the held-out test set.

Ridge Regression selected an alpha of 100.0. Its cross-validation MAE was 5987.33 and its held-out test MAE was 5944.91, with a test R² of 0.7010. Its performance was very similar to Linear Regression.

The Random Forest grid search required substantially more computation time than the linear models. Because the Random Forest GridSearchCV parallelization had to be reduced to two simultaneous jobs after the original run exceeded available Windows system resources, its grid-search wall-clock time should be interpreted cautiously.

To provide a more consistent measure of final model training time, each model was subsequently refitted once on the complete 94-feature training dataset using its selected hyperparameters. The final refit times were 0.57 seconds for Linear Regression, 57.25 seconds for Random Forest, and 0.20 seconds for Ridge Regression. These final refit times, rather than the GridSearchCV times, will be used when comparing training efficiency with the Notebook 3 models.

## 5. Retrain Keras Neural Networks

The three Keras neural network architectures evaluated in Notebook 3 are retrained using the reduced 94-feature dataset. The same architectures and training settings are maintained so that model performance before and after feature selection can be compared consistently.

### 5.1 Define Keras Model Architectures

Recreate the three neural network architectures from Notebook 3. All architectures use ReLU activation in the hidden layers, a single output neuron for price prediction, the Adam optimizer, and mean absolute error (MAE) as the loss function.

Each architecture uses the 94 features retained after Random Forest importance selection and VIF filtering as input. After defining the models, the input dimension and number of model parameters are verified before cross-validation.

In [27]:
# Architecture 1
def build_nn1(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )

    return model


# Architecture 2
def build_nn2(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )

    return model


# Architecture 3
def build_nn3(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.20),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.20),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )

    return model

In [28]:
# Verify that all three architectures build correctly
input_dim = X_train_selected.shape[1]

nn1_check = build_nn1(input_dim)
nn2_check = build_nn2(input_dim)
nn3_check = build_nn3(input_dim)

print("Number of selected input features:", input_dim)
print("Architecture 1 parameters:", nn1_check.count_params())
print("Architecture 2 parameters:", nn2_check.count_params())
print("Architecture 3 parameters:", nn3_check.count_params())

# Clear temporary models from memory
del nn1_check, nn2_check, nn3_check
tf.keras.backend.clear_session()

Number of selected input features: 94
Architecture 1 parameters: 20481
Architecture 2 parameters: 65537
Architecture 3 parameters: 65537



### 5.2 Cross-Validate Architecture 1

Evaluate Architecture 1 using manual 5-fold cross-validation on the reduced 94-feature training set. A fresh neural network is created for each fold to prevent information learned during one fold from carrying over into another.

The model is trained for a maximum of 50 epochs with a batch size of 256. Early stopping monitors validation loss with a patience of 5 epochs and restores the best model weights. MAE, MAPE, and R² are calculated for each validation fold and averaged to summarize cross-validation performance.

In [29]:
# Store Architecture 1 fold results
nn1_fold_mae = []
nn1_fold_mape = []
nn1_fold_r2 = []

# Convert to NumPy arrays for Keras cross-validation
X_nn = X_train_selected.to_numpy(dtype=np.float32)
y_nn = y_train.to_numpy(dtype=np.float32)

# Time the complete 5-fold CV process
nn1_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(cv.split(X_nn), start=1):

    print(f"\nArchitecture 1 - Fold {fold}/5")
    print("-" * 35)

    # Split current fold
    X_fold_train = X_nn[train_idx]
    X_fold_val = X_nn[val_idx]

    y_fold_train = y_nn[train_idx]
    y_fold_val = y_nn[val_idx]

    # Clear previous Keras model and set reproducible seed
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)

    # Create a fresh model for this fold
    model = build_nn1(X_nn.shape[1])

    # Early stopping
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # Train model
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    # Predict validation fold
    val_pred = model.predict(
        X_fold_val,
        batch_size=256,
        verbose=0
    ).ravel()

    # Calculate validation metrics
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)

    # Store fold results
    nn1_fold_mae.append(fold_mae)
    nn1_fold_mape.append(fold_mape)
    nn1_fold_r2.append(fold_r2)

    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")

# Total cross-validation time
nn1_cv_time = time.time() - nn1_start_time


Architecture 1 - Fold 1/5
-----------------------------------
Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 18719.6504 - mae: 18719.6504 - mape: 62.2930 - val_loss: 7134.9668 - val_mae: 7134.9668 - val_mape: 26.3945
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 5927.0811 - mae: 5927.0811 - mape: 21.1356 - val_loss: 5106.9048 - val_mae: 5106.9048 - val_mape: 17.7602
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 4788.1465 - mae: 4788.1465 - mape: 16.2785 - val_loss: 4487.2783 - val_mae: 4487.2783 - val_mape: 15.3335
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 4355.3848 - mae: 4355.3848 - mape: 14.6954 - val_loss: 4200.1636 - val_mae: 4200.1636 - val_mape: 14.4010
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 4134.4155 - mae: 4134.4155 - mape: 13.9666 - val_loss: 4035.3896 - val_mae: 4035.3896 - val_mape: 13.8939
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 4001.4014 - mae: 4001.4014 - mape: 13.5230 - 

In [30]:
# Calculate mean cross-validation metrics
nn1_cv_mae = np.mean(nn1_fold_mae)
nn1_cv_mape = np.mean(nn1_fold_mape)
nn1_cv_r2 = np.mean(nn1_fold_r2)

print("Architecture 1 Cross-Validation Results")
print("-" * 45)
print(f"Mean CV MAE: ${nn1_cv_mae:,.2f}")
print(f"Mean CV MAPE: {nn1_cv_mape:.2%}")
print(f"Mean CV R²: {nn1_cv_r2:.4f}")
print(f"Total CV Time: {nn1_cv_time:.2f} seconds")

Architecture 1 Cross-Validation Results
---------------------------------------------
Mean CV MAE: $2,884.19
Mean CV MAPE: 10.32%
Mean CV R²: 0.7942
Total CV Time: 303.16 seconds


### 5.3 Cross-Validate Architecture 2

Evaluate Architecture 2 using the same manual 5-fold cross-validation procedure used for Architecture 1. A fresh neural network is created for each fold and trained using the same maximum number of epochs, batch size, early stopping criteria, and evaluation metrics.

Architecture 2 contains three hidden Dense layers with 256, 128, and 64 neurons, respectively. Maintaining the same training procedure allows differences in performance to be attributed primarily to the change in neural network architecture.

In [31]:
# Store Architecture 2 fold results
nn2_fold_mae = []
nn2_fold_mape = []
nn2_fold_r2 = []

# Time the complete 5-fold CV process
nn2_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(cv.split(X_nn), start=1):

    print(f"\nArchitecture 2 - Fold {fold}/5")
    print("-" * 35)

    # Split current fold
    X_fold_train = X_nn[train_idx]
    X_fold_val = X_nn[val_idx]

    y_fold_train = y_nn[train_idx]
    y_fold_val = y_nn[val_idx]

    # Clear previous Keras model and set reproducible seed
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)

    # Create a fresh Architecture 2 model
    model = build_nn2(X_nn.shape[1])

    # Early stopping
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # Train model
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    # Predict validation fold
    val_pred = model.predict(
        X_fold_val,
        batch_size=256,
        verbose=0
    ).ravel()

    # Calculate validation metrics
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)

    # Store fold results
    nn2_fold_mae.append(fold_mae)
    nn2_fold_mape.append(fold_mape)
    nn2_fold_r2.append(fold_r2)

    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")

# Total cross-validation time
nn2_cv_time = time.time() - nn2_start_time


Architecture 2 - Fold 1/5
-----------------------------------
Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 9087.8525 - mae: 9087.8525 - mape: 31.1696 - val_loss: 4184.6182 - val_mae: 4184.6182 - val_mape: 14.3403
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3971.7258 - mae: 3971.7258 - mape: 13.4355 - val_loss: 3768.9939 - val_mae: 3768.9939 - val_mape: 12.9547
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3699.3240 - mae: 3699.3240 - mape: 12.5395 - val_loss: 3586.8667 - val_mae: 3586.8667 - val_mape: 12.2873
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3532.7012 - mae: 3532.7012 - mape: 12.0114 - val_loss: 3444.0793 - val_mae: 3444.0793 - val_mape: 11.8563
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3395.2803 - mae: 3395.2803 - mape: 11.5896 - val_loss: 3328.4900 - val_mae: 3328.4900 - val_mape: 11.5363
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3278.1548 - mae: 3278.1548 - mape: 11.2523 - va

In [32]:
# Calculate mean cross-validation metrics
nn2_cv_mae = np.mean(nn2_fold_mae)
nn2_cv_mape = np.mean(nn2_fold_mape)
nn2_cv_r2 = np.mean(nn2_fold_r2)

print("Architecture 2 Cross-Validation Results")
print("-" * 45)
print(f"Mean CV MAE: ${nn2_cv_mae:,.2f}")
print(f"Mean CV MAPE: {nn2_cv_mape:.2%}")
print(f"Mean CV R²: {nn2_cv_r2:.4f}")
print(f"Total CV Time: {nn2_cv_time:.2f} seconds")

Architecture 2 Cross-Validation Results
---------------------------------------------
Mean CV MAE: $2,451.99
Mean CV MAPE: 9.15%
Mean CV R²: 0.8447
Total CV Time: 439.35 seconds


### 5.4 Cross-Validate Architecture 3

Evaluate Architecture 3 using the same manual 5-fold cross-validation procedure used for Architectures 1 and 2.

Architecture 3 uses three hidden Dense layers with 256, 128, and 64 neurons and adds 20% Dropout after the first two hidden layers. The same folds, optimizer, loss function, maximum number of epochs, batch size, early stopping criteria, and evaluation metrics are maintained for a consistent comparison among the three architectures.

In [33]:
# Store Architecture 3 fold results
nn3_fold_mae = []
nn3_fold_mape = []
nn3_fold_r2 = []

# Time the complete 5-fold CV process
nn3_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(cv.split(X_nn), start=1):

    print(f"\nArchitecture 3 - Fold {fold}/5")
    print("-" * 35)

    # Split current fold
    X_fold_train = X_nn[train_idx]
    X_fold_val = X_nn[val_idx]

    y_fold_train = y_nn[train_idx]
    y_fold_val = y_nn[val_idx]

    # Clear previous Keras model and set reproducible seed
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)

    # Create a fresh Architecture 3 model
    model = build_nn3(X_nn.shape[1])

    # Early stopping
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # Train model
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    # Predict validation fold
    val_pred = model.predict(
        X_fold_val,
        batch_size=256,
        verbose=0
    ).ravel()

    # Calculate validation metrics
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)

    # Store fold results
    nn3_fold_mae.append(fold_mae)
    nn3_fold_mape.append(fold_mape)
    nn3_fold_r2.append(fold_r2)

    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")

# Total cross-validation time
nn3_cv_time = time.time() - nn3_start_time


Architecture 3 - Fold 1/5
-----------------------------------
Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 9321.6992 - mae: 9321.6992 - mape: 32.1677 - val_loss: 4246.9946 - val_mae: 4246.9946 - val_mape: 14.4805
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4377.3262 - mae: 4377.3262 - mape: 14.8873 - val_loss: 3846.7517 - val_mae: 3846.7517 - val_mape: 13.3211
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4156.9424 - mae: 4156.9424 - mape: 14.1758 - val_loss: 3684.2546 - val_mae: 3684.2546 - val_mape: 12.6621
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4034.7263 - mae: 4034.7263 - mape: 13.7823 - val_loss: 3579.0271 - val_mae: 3579.0271 - val_mape: 12.3716
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3952.3098 - mae: 3952.3098 - mape: 13.5254 - val_loss: 3504.0679 - val_mae: 3504.0679 - val_mape: 12.2296
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3884.6665 - mae: 3884.6665 - mape: 13.3108 - va

In [34]:
# Calculate mean cross-validation metrics
nn3_cv_mae = np.mean(nn3_fold_mae)
nn3_cv_mape = np.mean(nn3_fold_mape)
nn3_cv_r2 = np.mean(nn3_fold_r2)

print("Architecture 3 Cross-Validation Results")
print("-" * 45)
print(f"Mean CV MAE: ${nn3_cv_mae:,.2f}")
print(f"Mean CV MAPE: {nn3_cv_mape:.2%}")
print(f"Mean CV R²: {nn3_cv_r2:.4f}")
print(f"Total CV Time: {nn3_cv_time:.2f} seconds")

Architecture 3 Cross-Validation Results
---------------------------------------------
Mean CV MAE: $2,649.08
Mean CV MAPE: 9.90%
Mean CV R²: 0.8394
Total CV Time: 515.43 seconds


### 5.5 Select and Refit the Best Keras Architecture

Compare the mean cross-validation performance of the three Keras architectures. The architecture with the lowest mean cross-validation MAE is selected for final training on the complete reduced training dataset.

After selecting the architecture, a new model is created and refitted using all 94 selected features in the training set.

In [35]:
# Compare Keras cross-validation results
keras_cv_results = pd.DataFrame({
    "Architecture": [
        "Keras Architecture 1",
        "Keras Architecture 2",
        "Keras Architecture 3"
    ],
    "CV MAE": [
        nn1_cv_mae,
        nn2_cv_mae,
        nn3_cv_mae
    ],
    "CV MAPE": [
        nn1_cv_mape,
        nn2_cv_mape,
        nn3_cv_mape
    ],
    "CV R2": [
        nn1_cv_r2,
        nn2_cv_r2,
        nn3_cv_r2
    ]
}).set_index("Architecture")

display(keras_cv_results.round({
    "CV MAE": 2,
    "CV MAPE": 4,
    "CV R2": 4
}))

# Select architecture with lowest CV MAE
best_keras_architecture = keras_cv_results["CV MAE"].idxmin()

print("Selected architecture:", best_keras_architecture)

,CV MAE,CV MAPE,CV R2
Architecture,,,
Keras Architecture 1,2884.19,0.1032,0.7942
Keras Architecture 2,2451.99,0.0915,0.8447
Keras Architecture 3,2649.08,0.0990,0.8394


Selected architecture: Keras Architecture 2


In [36]:
# Clear previous Keras models from memory
tf.keras.backend.clear_session()
tf.random.set_seed(42)

# Build a fresh copy of the winning architecture
best_keras_selected = build_nn2(X_nn.shape[1])

# Time the final refit on the complete reduced training set
start_time = time.perf_counter()

history_selected = best_keras_selected.fit(
    X_nn,
    y_nn,
    epochs=50,
    batch_size=256,
    verbose=1
)

keras_selected_train_time = time.perf_counter() - start_time

print(
    f"Final selected Keras training time: "
    f"{keras_selected_train_time:.2f} seconds"
)

Epoch 1/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 7930.6533 - mae: 7930.6533 - mape: 27.3258
Epoch 2/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3826.3665 - mae: 3826.3665 - mape: 12.9976
Epoch 3/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3579.2793 - mae: 3579.2793 - mape: 12.1898
Epoch 4/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3415.6931 - mae: 3415.6931 - mape: 11.6763
Epoch 5/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3277.9519 - mae: 3277.9519 - mape: 11.2628
Epoch 6/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3162.3469 - mae: 3162.3469 - mape: 10.9280
Epoch 7/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3065.5085 - mae: 3065.5085 - mape: 10.6509
Epoch 8/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2982.1946 - mae: 2982.1946 - mape: 10.4184
Epoch 9/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2908.3137 - mae: 2908.3137 - mape: 10.2081
Epoch 10/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2844.9167 - 

### 5.6 Evaluate the Final Keras Model

Evaluate the final Architecture 2 model on the held-out test set. The test set was not used during model training or cross-validation.

Performance is measured using MAE, MAPE, and R² so that the selected-feature Keras model can be compared directly with the original Notebook 3 model and the retrained scikit-learn models.

In [37]:
# Convert selected test features to NumPy for Keras
X_test_nn = X_test_selected.to_numpy(dtype=np.float32)
y_test_nn = y_test.to_numpy(dtype=np.float32)

# Generate predictions on the held-out test set
keras_selected_pred = best_keras_selected.predict(
    X_test_nn,
    batch_size=256,
    verbose=0
).ravel()

# Calculate held-out test metrics
keras_selected_test_mae = mean_absolute_error(
    y_test_nn,
    keras_selected_pred
)

keras_selected_test_mape = mean_absolute_percentage_error(
    y_test_nn,
    keras_selected_pred
)

keras_selected_test_r2 = r2_score(
    y_test_nn,
    keras_selected_pred
)

print("Keras Architecture 2 - Selected Features")
print("-" * 45)
print(f"Test MAE: ${keras_selected_test_mae:,.2f}")
print(f"Test MAPE: {keras_selected_test_mape:.2%}")
print(f"Test R²: {keras_selected_test_r2:.4f}")
print(f"Final Training Time: {keras_selected_train_time:.2f} seconds")

Keras Architecture 2 - Selected Features
---------------------------------------------
Test MAE: $2,341.08
Test MAPE: 8.95%
Test R²: 0.9136
Final Training Time: 85.06 seconds


### Section 5 Summary

All three Keras neural network architectures from Notebook 3 were retrained and evaluated using the reduced 94-feature dataset.

Architecture 1 produced a mean cross-validation MAE of $2,877.67, MAPE of 10.29%, and R² of 0.7944. Architecture 2 improved performance with a mean cross-validation MAE of $2,455.78, MAPE of 9.20%, and R² of 0.8431. Architecture 3 produced a mean cross-validation MAE of $2,666.54, MAPE of 9.93%, and R² of 0.8406.

Architecture 2 achieved the lowest cross-validation MAE and MAPE and the highest cross-validation R² among the three architectures. Therefore, Architecture 2 was selected as the final Keras model.

A fresh Architecture 2 model was then trained on the complete reduced training dataset for 50 epochs. The final model trained in 93.90 seconds. On the held-out test set, it achieved an MAE of $2,375.17, MAPE of 9.02%, and R² of 0.9127.

These results will be compared with the original Notebook 3 results in the before-and-after feature-selection comparison.

## 6. Before/After Model Performance Comparison

### 6.1 Create the Selected-Feature Performance Table

The final performance results from the models trained on the 94 selected features are combined into a table using the same structure as the Notebook 3 baseline performance table. This allows the models to be compared directly before and after feature selection.

For training time, the final refit time is used rather than the total GridSearchCV runtime so that training efficiency is measured consistently.

In [38]:
# Create final performance table for models trained on selected features

selected_metrics = pd.DataFrame(
    {
        "CV MAE": [
            5991.016587,
            1799.816284,
            5987.331618,
            2455.78
        ],
        "CV MAPE": [
            0.288787,
            0.074237,
            0.288532,
            0.0920
        ],
        "CV R2": [
            0.651157,
            0.898400,
            0.651159,
            0.8431
        ],
        "Test MAE": [
            5947.921657,
            1701.087367,
            5944.907367,
            keras_selected_test_mae
        ],
        "Test MAPE": [
            0.291266,
            0.071589,
            0.291065,
            keras_selected_test_mape
        ],
        "Test R2": [
            0.700956,
            0.947267,
            0.700997,
            keras_selected_test_r2
        ],
        "Training Time (sec)": [
            linear_selected_train_time,
            rf_selected_train_time,
            ridge_selected_train_time,
            keras_selected_train_time
        ]
    },
    index=[
        "Linear Regression",
        "Random Forest",
        "Ridge Regression",
        "Keras Architecture 2"
    ]
)

selected_metrics.index.name = "Model"

print("Model Performance After Feature Selection")
print("-" * 50)

display(selected_metrics.round(4))

Model Performance After Feature Selection
--------------------------------------------------


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5991.0166,0.2888,0.6512,5947.9217,0.2913,0.7010,0.3733
Random Forest,1799.8163,0.0742,0.8984,1701.0874,0.0716,0.9473,56.1927
Ridge Regression,5987.3316,0.2885,0.6512,5944.9074,0.2911,0.7010,0.2125
Keras Architecture 2,2455.7800,0.0920,0.8431,2341.0791,0.0895,0.9136,85.0622


### 6.2 Compare Performance Before and After Feature Selection

The selected-feature model results are compared directly with the baseline results from Notebook 3. For each model, the change in held-out test MAE, MAPE, R², and final training time is calculated.

For MAE and MAPE, lower values indicate better predictive performance. For R², higher values indicate better predictive performance. Training-time reduction represents the percentage decrease in final model refit time after feature selection.

In [39]:
# Compare Notebook 3 baseline results with Notebook 4 selected-feature results

comparison = pd.DataFrame(index=baseline_metrics.index)

# Baseline and selected-feature test performance
comparison["Before Test MAE"] = baseline_metrics["Test MAE"]
comparison["After Test MAE"] = selected_metrics["Test MAE"]

comparison["Before Test MAPE"] = baseline_metrics["Test MAPE"]
comparison["After Test MAPE"] = selected_metrics["Test MAPE"]

comparison["Before Test R2"] = baseline_metrics["Test R2"]
comparison["After Test R2"] = selected_metrics["Test R2"]

# Final refit training times
comparison["Before Time (sec)"] = baseline_metrics["Training Time (sec)"]
comparison["After Time (sec)"] = selected_metrics["Training Time (sec)"]

# Calculate changes in predictive performance
comparison["MAE Change"] = (
    comparison["After Test MAE"] -
    comparison["Before Test MAE"]
)

comparison["MAPE Change"] = (
    comparison["After Test MAPE"] -
    comparison["Before Test MAPE"]
)

comparison["R2 Change"] = (
    comparison["After Test R2"] -
    comparison["Before Test R2"]
)

# Calculate percentage reduction in final refit training time
comparison["Time Reduction (%)"] = (
    (
        comparison["Before Time (sec)"] -
        comparison["After Time (sec)"]
    )
    / comparison["Before Time (sec)"]
) * 100

comparison.index.name = "Model"

print("Before vs. After Feature Selection")
print("-" * 50)

display(comparison.round(4))

Before vs. After Feature Selection
--------------------------------------------------


,Before Test MAE,After Test MAE,Before Test MAPE,After Test MAPE,Before Test R2,After Test R2,Before Time (sec),After Time (sec),MAE Change,MAPE Change,R2 Change,Time Reduction (%)
Model,,,,,,,,,,,,
Linear Regression,5498.2234,5947.9217,0.2686,0.2913,0.7339,0.7010,2.3710,0.3733,449.6983,0.0226,-0.0330,84.2569
Random Forest,1638.9686,1701.0874,0.0692,0.0716,0.9503,0.9473,122.2486,56.1927,62.1188,0.0024,-0.0030,54.0340
Ridge Regression,5494.7594,5944.9074,0.2684,0.2911,0.7340,0.7010,2.2632,0.2125,450.1480,0.0227,-0.0330,90.6105
Keras Architecture 2,2039.4022,2341.0791,0.0772,0.0895,0.9263,0.9136,122.7268,85.0622,301.6769,0.0123,-0.0127,30.6898


### 6.3 Interpretation of Accuracy and Training-Speed Changes

Feature selection reduced the modeling dataset from 293 features to 94 features, a reduction of approximately 67.9%. This substantially reduced final model training time, although some predictive performance was lost.

**Linear Regression:**  
The held-out test MAE increased from $5,498.22 to $5,947.92, an increase of $449.70. Test MAPE increased from 26.86% to 29.13%, while test R² decreased from 0.7339 to 0.7010. However, final refit time decreased from 2.37 seconds to 0.57 seconds, a 76.1% reduction. Therefore, feature selection substantially improved training speed but reduced predictive accuracy for Linear Regression.

**Random Forest:**  
The held-out test MAE increased from $1,638.97 to $1,701.09, an increase of only $62.12. Test MAPE increased slightly from 6.92% to 7.16%, and test R² decreased only slightly from 0.9503 to 0.9473. At the same time, final refit time decreased from 122.25 seconds to 57.25 seconds, a 53.2% reduction. Random Forest therefore retained most of its predictive performance while requiring substantially less final training time.

**Ridge Regression:**  
The held-out test MAE increased from $5,494.76 to $5,944.91, an increase of $450.15. Test MAPE increased from 26.84% to 29.11%, and test R² decreased from 0.7340 to 0.7010. Final refit time decreased from 2.26 seconds to 0.20 seconds, a 91.3% reduction. Similar to Linear Regression, Ridge Regression gained substantial computational efficiency at the cost of reduced predictive accuracy.

**Keras Architecture 2:**  
The held-out test MAE increased from $2,039.40 to $2,375.17, an increase of $335.76. Test MAPE increased from 7.72% to 9.02%, while test R² decreased from 0.9263 to 0.9127. Final training time decreased from 122.73 seconds to 93.90 seconds, a 23.5% reduction. Thus, the reduced feature set improved training efficiency for the neural network, but the decrease in predictive performance was greater than that observed for Random Forest.

Overall, feature selection produced a trade-off between computational efficiency and predictive performance. All four models trained faster with the reduced 94-feature dataset, but all also experienced higher prediction errors and lower test R² values. Among the evaluated models, Random Forest showed the smallest loss in held-out predictive performance while reducing its final refit time by more than half.

### Section 6 Summary

The before-and-after comparison showed that reducing the feature set from 293 to 94 features improved final training efficiency for every evaluated model. Training-time reductions ranged from 23.5% for Keras Architecture 2 to 91.3% for Ridge Regression.

The improvement in computational efficiency was accompanied by some loss in predictive performance. Linear Regression and Ridge Regression experienced the largest decreases in test R², while Keras Architecture 2 showed a moderate decrease. Random Forest was the least affected, with test MAE increasing by only $62.12 and test R² decreasing from 0.9503 to 0.9473 while final refit time decreased by 53.2%.

These results demonstrate the trade-off introduced by feature selection: a substantially smaller feature set and faster model fitting were achieved at the cost of some predictive accuracy.

## 7. Export Selected Data, Metrics, and Models

### 7.1 Create Selected Training and Test Datasets

The final selected training and test datasets are constructed using the 94 features retained after Random Forest importance selection and VIF filtering. The target variable (`price`) and all original object/string metadata columns are also preserved so that the selected datasets can be used in Notebook 5 for explainability and fairness analysis.

The expected number of columns in each exported dataset is 113: 94 selected modeling features, 18 metadata columns, and the target variable.

In [40]:
# Build selected datasets for export

# Get the final 94 selected feature names directly
# from the reduced training feature matrix
final_features = X_train_selected.columns.tolist()

# Preserve the original column order where possible
selected_export_cols = [
    col for col in train_df.columns
    if col in final_features
    or col in metadata_cols
    or col == target
]

selected_train = train_df[selected_export_cols].copy()
selected_test = test_df[selected_export_cols].copy()

print("Selected Dataset Summary")
print("-" * 40)

print("Selected modeling features:", len(final_features))
print("Metadata columns:", len(metadata_cols))
print("Target columns:", 1)

print("\nExpected total columns:",
      len(final_features) + len(metadata_cols) + 1)

print("Selected train shape:", selected_train.shape)
print("Selected test shape:", selected_test.shape)

print("\nTrain/test columns match:",
      list(selected_train.columns) == list(selected_test.columns))

Selected Dataset Summary
----------------------------------------
Selected modeling features: 94
Metadata columns: 18
Target columns: 1

Expected total columns: 113
Selected train shape: (237268, 113)
Selected test shape: (59317, 113)

Train/test columns match: True


### 7.2 Export Selected Training and Test Datasets

The final selected training and test datasets are exported for use in Notebook 5. Each dataset contains the 94 selected modeling features, the target variable (`price`), and all 18 preserved object/string metadata columns.

In [41]:
# Export selected training and test datasets

selected_train_path = data_dir / "selected_train.csv"
selected_test_path = data_dir / "selected_test.csv"

selected_train.to_csv(selected_train_path, index=False)
selected_test.to_csv(selected_test_path, index=False)

print("Selected datasets exported successfully.")
print("-" * 40)

print("Training data:", selected_train_path.resolve())
print("Test data:", selected_test_path.resolve())

print("\nTraining file exists:", selected_train_path.exists())
print("Test file exists:", selected_test_path.exists())


Selected datasets exported successfully.
----------------------------------------
Training data: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\selected_train.csv
Test data: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\selected_test.csv

Training file exists: True
Test file exists: True


### 7.3 Export Selected-Feature Model Metrics

The final performance metrics for the models trained on the 94 selected features are exported to `model_metrics_selected.csv`. The exported table uses the same performance measures as the Notebook 3 baseline table, allowing the original and selected-feature model results to be compared consistently.

In [42]:
# Export selected-feature model performance metrics

selected_metrics_path = data_dir / "model_metrics_selected.csv"

selected_metrics.to_csv(selected_metrics_path, index=True)

print("Selected-feature model metrics exported successfully.")
print("-" * 50)

print("Metrics file:", selected_metrics_path.resolve())
print("File exists:", selected_metrics_path.exists())

print("\nExported metrics shape:", selected_metrics.shape)

display(selected_metrics.round(4))

Selected-feature model metrics exported successfully.
--------------------------------------------------
Metrics file: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\model_metrics_selected.csv
File exists: True

Exported metrics shape: (4, 7)


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5991.0166,0.2888,0.6512,5947.9217,0.2913,0.7010,0.3733
Random Forest,1799.8163,0.0742,0.8984,1701.0874,0.0716,0.9473,56.1927
Ridge Regression,5987.3316,0.2885,0.6512,5944.9074,0.2911,0.7010,0.2125
Keras Architecture 2,2455.7800,0.0920,0.8431,2341.0791,0.0895,0.9136,85.0622


### 7.4 Save Selected-Feature Models

The final models trained on the 94 selected features are saved for later use. The scikit-learn models are exported using `joblib`, while the selected Keras neural network is saved in the native Keras format. The filenames use the `_selected` suffix to distinguish these models from the full-feature models created in Notebook 3.

In [43]:
# Define the project models directory
models_dir = Path("../../models")

print("Models directory:", models_dir.resolve())
print("Directory exists:", models_dir.exists())

Models directory: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\models
Directory exists: True


In [44]:
# Save final selected-feature models

linear_selected_path = models_dir / "P1_N4_linreg_selected.pkl"
rf_selected_path = models_dir / "P1_N4_rf_selected.pkl"
ridge_selected_path = models_dir / "P1_N4_ridge_selected.pkl"
keras_selected_path = models_dir / "P1_N4_nn2_selected.keras"

# Save scikit-learn models
joblib.dump(linear_selected, linear_selected_path)
joblib.dump(rf_selected, rf_selected_path)
joblib.dump(ridge_selected, ridge_selected_path)

# Save selected Keras model
best_keras_selected.save(keras_selected_path)

print("Selected-feature models saved successfully.")
print("-" * 50)

print("Linear Regression:", linear_selected_path.resolve())
print("Random Forest:", rf_selected_path.resolve())
print("Ridge Regression:", ridge_selected_path.resolve())
print("Keras Architecture 2:", keras_selected_path.resolve())

print("\nFile checks:")
print("Linear Regression:", linear_selected_path.exists())
print("Random Forest:", rf_selected_path.exists())
print("Ridge Regression:", ridge_selected_path.exists())
print("Keras Architecture 2:", keras_selected_path.exists())

Selected-feature models saved successfully.
--------------------------------------------------
Linear Regression: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\models\P1_N4_linreg_selected.pkl
Random Forest: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\models\P1_N4_rf_selected.pkl
Ridge Regression: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\models\P1_N4_ridge_selected.pkl
Keras Architecture 2: C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\models\P1_N4_nn2_selected.keras

File checks:
Linear Regression: True
Random Forest: True
Ridge Regression: True
Keras Architecture 2: True


### 7.5 Final Export Verification

A final verification is performed to confirm that all datasets, performance metrics, and selected-feature models required from Notebook 4 were successfully exported to the appropriate project directories.

In [45]:
# Final Notebook 4 export verification

files_to_check = {
    "Selected training data":
        data_dir / "selected_train.csv",

    "Selected test data":
        data_dir / "selected_test.csv",

    "Selected model metrics":
        data_dir / "model_metrics_selected.csv",

    "Linear Regression":
        models_dir / "P1_N4_linreg_selected.pkl",

    "Random Forest":
        models_dir / "P1_N4_rf_selected.pkl",

    "Ridge Regression":
        models_dir / "P1_N4_ridge_selected.pkl",

    "Keras Architecture 2":
        models_dir / "P1_N4_nn2_selected.keras"
}

print("FINAL NOTEBOOK 4 EXPORT CHECK")
print("-" * 55)

for name, path in files_to_check.items():
    print(f"{name:<28} {path.exists()}")

print("\nSelected dataset verification")
print("-" * 55)

check_train = pd.read_csv(data_dir / "selected_train.csv")
check_test = pd.read_csv(data_dir / "selected_test.csv")

print("Selected train shape:", check_train.shape)
print("Selected test shape:", check_test.shape)
print("Train/test columns match:",
      list(check_train.columns) == list(check_test.columns))

print("\nSelected metrics verification")
print("-" * 55)

check_metrics = pd.read_csv(
    data_dir / "model_metrics_selected.csv",
    index_col="Model"
)

print("Metrics shape:", check_metrics.shape)
display(check_metrics.round(4))

FINAL NOTEBOOK 4 EXPORT CHECK
-------------------------------------------------------
Selected training data       True
Selected test data           True
Selected model metrics       True
Linear Regression            True
Random Forest                True
Ridge Regression             True
Keras Architecture 2         True

Selected dataset verification
-------------------------------------------------------
Selected train shape: (237268, 113)
Selected test shape: (59317, 113)
Train/test columns match: True

Selected metrics verification
-------------------------------------------------------
Metrics shape: (4, 7)


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5991.0166,0.2888,0.6512,5947.9217,0.2913,0.7010,0.3733
Random Forest,1799.8163,0.0742,0.8984,1701.0874,0.0716,0.9473,56.1927
Ridge Regression,5987.3316,0.2885,0.6512,5944.9074,0.2911,0.7010,0.2125
Keras Architecture 2,2455.7800,0.0920,0.8431,2341.0791,0.0895,0.9136,85.0622


### Section 7 Summary

The final outputs from Notebook 4 were successfully exported and verified. The selected training and test datasets each contain 94 modeling features, 18 preserved metadata columns, and the target variable (`price`), for a total of 113 columns.

The selected-feature model performance table was exported as `model_metrics_selected.csv`. The final Linear Regression, Random Forest, Ridge Regression, and Keras Architecture 2 models were also saved with `_selected` filenames for later use.

A final verification confirmed that all required output files exist, the selected training and test datasets have matching columns, and the exported model metrics contain results for all four final models.